# Reto II NLP Avanzado

- Pueden encontrar los datos de este reto en esta liga. Posiblemente los estudiantes no estén familiarizados con la política interir en la Argentina.
- La intención de este reto es utilizar Latent Dirichlet Allocation junto a la API de ChatGPT para construir un Topic Modelling. Recuerden que aquí están las instrucciones para hacer uso del API.
- El primer paso será construir un Topic Modeling mediante LDA para después extraer el significado de los tópicos utilizando LDA. Es necesario comparar dos enfoques posibles para extraer el significado de estos tópicos:
1) Como lo hicimos en clase utilizando los pesos de las palabras más
relevantes en el tópico.
2) Alimentando a ChatGPT con los textos que tengan mayor
presencia del tópico mencionado.

## Enfoque 1: Pesos en las palabras más relevantes en el tópico

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# -----------------------
# 1) Load dataset
# -----------------------
PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_DIR = PROJECT_ROOT / "data" / "input" / "MM"
txt_files = sorted(DATA_DIR.rglob("*.txt"))

rows = []
for i, fp in enumerate(txt_files, start=1):
    try:
        text = fp.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = fp.read_text(encoding="latin-1", errors="replace")

    rows.append({
        "doc_id": i,
        "filename": fp.name,
        "rel_path": str(fp.relative_to(PROJECT_ROOT)),
        "text": text.strip(),
    })

df = pd.DataFrame(rows)
df.head()

,doc_id,filename,rel_path,text
0,1,MM_01_03_2019_0.txt,data/input/MM/MM_01_03_2019_0.txt,"Gracias por las lindas palabras, feliz año, fe..."
1,2,MM_01_04_2016_0.txt,data/input/MM/MM_01_04_2016_0.txt,"Buen día a todos. Gracias Carlos, gracias Inte..."
2,3,MM_01_05_2016_0.txt,data/input/MM/MM_01_05_2016_0.txt,"Buenos días. Como decía Horacio, qué bueno que..."
3,4,MM_01_06_2016_0.txt,data/input/MM/MM_01_06_2016_0.txt,Buenas tardes.\n\nEs una alegría por múltiples...
4,5,MM_01_08_2016_0.txt,data/input/MM/MM_01_08_2016_0.txt,"Buenos días, ¿cómo estamos hoy acá en Santa Fe..."


In [3]:
# Para replicabilidad
RANDOM_SEED = 42

In [7]:
# -----------------------
# 2) Filter: Spanish root prompts from the user
# -----------------------
docs_raw = df["text"].astype(str).tolist()
print("Docs (raw):", len(docs_raw))

Docs (raw): 663


In [10]:
# -----------------------
# 3) Preprocessing (standard NLP pipeline + stemming)
#    clean -> normalize -> tokenize -> stopwords -> stem -> re-join
# -----------------------
import unicodedata
import re
from collections import Counter

# Stemming (Spanish)
from nltk.stem.snowball import SnowballStemmer
import warnings
warnings.filterwarnings("ignore")

def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

STOPWORDS_ES = {
    # common Spanish
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","no",
    "una","su","al","lo","como","mas","pero","sus","le","ya","o","este","si","porque",
    "esta","entre","cuando","muy","sin","sobre","tambien","me","hasta","hay","donde",
    "quien","desde","todo","nos","durante","todos","uno","les","ni","contra","otros",

    # question/prompt words (important for chatbot prompts)
    "que","como","cual","cuales","quien","donde","cuando","cuanto","cuantos","por","porque",

    # prompt fluff
    "hola","buenas","gracias","porfavor","favor",
    "puedes","podrias","ayudame","necesito","dime","explica","explicame","describe","resume",
    "haz","dame","escribe","crea","genera",

    # common high-frequency verbs/forms
    "ser","estar","tener","hacer","poder",
    "soy","eres","es","son","estoy","esta","estan","tengo","tiene","tienen","quiero","puedo",
}

# Normalize stopwords in the same way we normalize text
STOPWORDS_ES = {strip_accents(w.lower()) for w in STOPWORDS_ES}

# Spanish stemmer (simple + no extra downloads)
stemmer = SnowballStemmer("spanish")

def preprocess(text: str) -> str:
    # A) CLEANING (remove obvious noise)
    text = (text or "").lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # remove URLs

    # B) NORMALIZATION (accent folding)
    text = strip_accents(text)  # "qué" -> "que"

    # C) TOKENIZATION (letters only; after accent stripping)
    tokens = re.findall(r"[a-zñ]+", text)

    # D) FILTERING (stopwords + short tokens)
    tokens = [t for t in tokens if len(t) >= 3 and t not in STOPWORDS_ES]

    # E) STEMMING (reduce words to their root)
    tokens = [stemmer.stem(t) for t in tokens]

    # Return a string for CountVectorizer
    return " ".join(tokens)

docs_clean = [preprocess(t) for t in docs_raw]

# Keep alignment between raw and clean by filtering pairs together
pairs = [(r, c) for r, c in zip(docs_raw, docs_clean) if c.strip()]
docs_raw, docs_clean = zip(*pairs) if pairs else ([], [])
docs_raw, docs_clean = list(docs_raw), list(docs_clean)

print("Docs (clean, non-empty):", len(docs_clean))
print("Top tokens after preprocessing:", Counter(" ".join(docs_clean).split()).most_common(20))

Docs (clean, non-empty): 663
Top tokens after preprocessing: [('argentin', 6032), ('trabaj', 4559), ('eso', 3777), ('nuestr', 3707), ('ten', 3558), ('estam', 3269), ('much', 3214), ('hac', 3152), ('pais', 2863), ('cad', 2427), ('anos', 2277), ('hoy', 2222), ('hem', 2025), ('usted', 2019), ('mund', 1932), ('aca', 1859), ('esto', 1859), ('pued', 1837), ('cre', 1836), ('vam', 1829)]


In [13]:
# -----------------------
# 4) Bag-of-words
# -----------------------
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2))
X = vectorizer.fit_transform(docs_clean)
vocab = np.array(vectorizer.get_feature_names_out())
print("Vocab size:", len(vocab))

Vocab size: 52399


In [14]:
# -----------------------
# 5) Fit LDA
# -----------------------
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 12
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method="batch",
    random_state=RANDOM_SEED,
    max_iter=25
)
lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",12
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",25
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [23]:
# -----------------------
# 6) Topics: print + DataFrame of top words per topic
# -----------------------
def show_topics(model, vocab, top_n=12):
    for k, weights in enumerate(model.components_):
        top_idx = np.argsort(weights)[::-1][:top_n]
        top_terms = [vocab[i] for i in top_idx]
        print(f"Topic {k}: {', '.join(top_terms)}")

def topics_dataframe(model, vocab, top_n=12):
    rows = []
    for topic_id, weights in enumerate(model.components_):
        top_idx = np.argsort(weights)[::-1][:top_n]
        for rank, i in enumerate(top_idx, start=1):
            rows.append({
                "topic_id": topic_id,
                "rank": rank,
                "term": vocab[i],
                "weight": float(weights[i]),
            })
    return pd.DataFrame(rows)

show_topics(lda, vocab, top_n=20)

df_topics = topics_dataframe(lda, vocab, top_n=20)

Topic 0: cad, pais, vam, mejor, buen, ese, anos, hem, hoy, junt, cre, import, pas, aca, gener, mund, camin, dia, pued, esto
Topic 1: estad, gobiern, vam, esto, public, inform, pais, hoy, cad, part, tem, ese, llev, provinci, buen, mejor, tod, transparent, hem, pued
Topic 2: president, pais, mund, desarroll, oportun, visit, hem, nuestr pais, relacion, cre, integr, import, tem, internacional, mercosur, futur, anos, hoy, acuerd, asi
Topic 3: pais, mund, cad, anos, hem, vam, usted, pas, cos, hoy, import, cre, esto, quer, cambi, part, verd, pued, aca, buen
Topic 4: franci, president, pais, quer, part, mund, usted, pod, hech, dij, cre, cos, hem, dec, pued, frances, mejor, vez, sab, realment
Topic 5: hoy, cad, pais, aca, anos, mejor, hem, ese, usted, mund, import, cos, part, cre, esto, dia, vam, pas, pued, quer
Topic 6: pais, cad, anos, pued, hoy, esto, aca, vam, usted, mejor, pas, buen, cre, hem, quer, import, mund, junt, vez, ese
Topic 7: cobr, jubil, sentenci, situacion, juici, sentenci fir

In [24]:
df_topics

,topic_id,rank,term,weight
0,0,1,cad,371.306112
1,0,2,pais,269.196793
2,0,3,vam,223.772888
3,0,4,mejor,214.866257
4,0,5,buen,209.684789
...,...,...,...,...
235,11,16,ese,249.857059
236,11,17,pas,247.770285
237,11,18,quer,238.272095
238,11,19,mund,234.641446


In [26]:
# -----------------------
# 7) Documents: DataFrame with topic assignment + probabilities
# -----------------------
from textwrap import shorten

doc_topic = lda.transform(X)
top_topic = doc_topic.argmax(axis=1)
top_conf = doc_topic.max(axis=1)

df_docs = pd.DataFrame({
    "doc_id": np.arange(len(docs_raw)),
    "text_raw": [shorten(t, width=160, placeholder="…") for t in docs_raw],
    "text_clean": [shorten(t, width=160, placeholder="…") for t in docs_clean],
    "top_topic": top_topic,
    "top_topic_conf": np.round(top_conf, 3),
})

for k in range(n_topics):
    df_docs[f"topic_{k}_prob"] = np.round(doc_topic[:, k], 3)

df_docs

,doc_id,text_raw,text_clean,top_topic,top_topic_conf,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob,topic_10_prob,topic_11_prob
0,0,"Gracias por las lindas palabras, feliz año, fe...",lind palabr feliz ano feliz ano realment teng ...,3,0.986,0.000,0.000,0.013,0.986,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,1,"Buen día a todos. Gracias Carlos, gracias Inte...",buen dia carl intendent darn bienven gobern re...,6,0.662,0.000,0.000,0.053,0.000,0.000,0.000,0.662,0.000,0.000,0.000,0.284,0.000
2,2,"Buenos días. Como decía Horacio, qué bueno que...",buen dias deci horaci buen estem aca manan dan...,6,0.999,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.000,0.000,0.000,0.000,0.000
3,3,Buenas tardes. Es una alegría por múltiples ra...,tard alegri multipl razon primer recib amig di...,0,0.998,0.998,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,4,"Buenos días, ¿cómo estamos hoy acá en Santa Fe...",buen dias estam hoy aca sant ven tuert habr ve...,6,0.535,0.000,0.000,0.000,0.463,0.000,0.000,0.535,0.000,0.000,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658,658,"Buen día, gracias por acompañarnos. Estamos mu...",buen dia acompan estam content trabaj junt nue...,6,0.806,0.000,0.000,0.000,0.000,0.000,0.000,0.806,0.000,0.191,0.000,0.000,0.000
659,659,"BORDET.- Señor Presidente, quiero agradecerle ...",bordet senor president agradec presenci moment...,10,0.999,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.000
660,660,Buenos días a todos los entrerrianos que nos r...,buen dias entrerrian recib hoy aca buen dias a...,10,0.999,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.000
661,661,Buenos días a todos. Estoy estrenando mi nueva...,buen dias estren nuev voz despu oper bastant b...,3,0.372,0.320,0.000,0.000,0.372,0.000,0.000,0.170,0.000,0.000,0.000,0.137,0.000


In [27]:
# -----------------------
# 8) pyLDAvis
# -----------------------
import pyLDAvis
from IPython.display import display

pyLDAvis.enable_notebook()

topic_term_dists = lda.components_ / lda.components_.sum(axis=1)[:, None]
doc_topic_dists = doc_topic
doc_lengths = np.asarray(X.sum(axis=1)).ravel()
term_frequency = np.asarray(X.sum(axis=0)).ravel()

vis = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    sort_topics=False
)

display(vis)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.086996  0.053240       1        1  10.068936
1      0.034294  0.006663       2        1   2.109714
2     -0.095885 -0.205268       3        1   7.436314
3     -0.092284  0.024311       4        1  14.457870
4      0.087233 -0.051107       5        1   1.557575
5     -0.086635  0.048119       6        1   9.547890
6     -0.112166  0.034192       7        1  32.337425
7      0.255349  0.008859       8        1   0.305429
8      0.143595  0.033929       9        1   1.061240
9      0.042823 -0.081387      10        1   2.397380
10     0.002472  0.080295      11        1   3.693470
11    -0.091800  0.048154      12        1  15.026759, topic_info=            Term         Freq        Total Category  logprob  loglift
38243  president  1082.000000  1082.000000  Default  30.0000  30.0000
35115       pais  2656.000000  2656.000000  Default  29.0000  29.0000
19304      estad   798.000000   798.000000  Default  28.0000  28.0000
25375        hoy  2065.000000  2065.000000  Default  27.0000  27.0000
18182        ese  1495.000000  1495.000000  Default  26.0000  26.0000
...          ...          ...          ...      ...      ...      ...
49841        vam   236.940016  1707.374643  Topic12  -5.8269  -0.0796
17832        esa   195.501750  1179.756931  Topic12  -6.0191   0.0978
32421       mund   222.185384  1798.367543  Topic12  -5.8912  -0.1958
11152        cre   218.312159  1707.860273  Topic12  -5.9087  -0.1617
13999        dia   194.963450  1245.121840  Topic12  -6.0219   0.0412

[1024 rows x 6 columns], token_table=       Topic      Freq          Term
term                                
25         9  0.521739  abandon tare
43         1  0.053618        abiert
43         2  0.130214        abiert
43         3  0.176172        abiert
43         4  0.199151        abiert
...      ...       ...           ...
52382      1  0.042504           ypf
52382      4  0.680066           ypf
52382      6  0.042504           ypf
52382      7  0.170017           ypf
52382     12  0.042504           ypf

[2882 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])

In [ ]:
# -----------------------
# 8) LLM (OpenAI) Juez
# -----------------------
system_message = (
    "Eres un generador de etiquetas de temas en español. "
    "Tu unica salida debe ser UNA SOLA LINEA con la etiqueta del tema, "
    "en minúsculas, máximo 3 palabras, sin puntuación final, sin comillas y sin explicaciones."
)

# prepare topic metadata: one label per topic_id + up to 10 unique terms
topic_terms = (
    
)